# Box Point Inspector

Wähle eine Box aus der Trajektorie und visualisiere alle Zielpunkte, die entlang der z-Achse innerhalb der rechteckigen Fläche liegen.

In [1]:
import numpy as np
import dill
from pathlib import Path
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

trajectory_path = Path("gt_coveragewithTargets_withBoxes.dill")  # Trajektorie mit Box- und TargetPoint-Metadaten


In [2]:
with open(trajectory_path, "rb") as f:
    trajectory = dill.load(f)

poses = np.array([pose[:3, 3] for pose in trajectory.poses_se3])
boxes = trajectory.meta.get("TargetBoxes", [])
target_points = trajectory.meta.get("TargetPoints", [])

print(f"Posen: {len(poses)}  |  Boxen: {len(boxes)}  |  TargetPoints: {len(target_points)}")
if not boxes:
    raise ValueError("Keine TargetBoxes in trajectory.meta gefunden. Bitte zuerst das Box-Notebook ausführen.")


Posen: 2741  |  Boxen: 119  |  TargetPoints: 600


In [3]:
def oriented_basis(box):
    direction = np.array(box["direction"], dtype=float)
    direction[2] = 0.0
    dir_norm = np.linalg.norm(direction[:2])
    if dir_norm < 1e-8:
        direction = np.array([1.0, 0.0, 0.0])
    else:
        direction /= dir_norm
    lateral = np.array([-direction[1], direction[0], 0.0])
    return direction, lateral


def filter_points_in_box(points, box):
    if points is None or len(points) == 0:
        return np.empty((0, 3))
    center = np.array(box["center"], dtype=float)
    length = float(box["length"])
    width = float(box["width"])
    direction, lateral = oriented_basis(box)
    rel = np.asarray(points) - center
    proj_forward = rel @ direction
    proj_side = rel @ lateral
    mask = (np.abs(proj_forward) <= length / 2.0) & (np.abs(proj_side) <= width / 2.0)
    selected = np.asarray(points)[mask]
    if selected.size:
        selected[:, 2] = center[2]
    return selected


def first_pose_index_in_box(poses, box):
    if poses is None or len(poses) == 0:
        return None, None
    center = np.array(box["center"], dtype=float)
    length = float(box["length"])
    width = float(box["width"])
    direction, lateral = oriented_basis(box)
    rel = poses - center
    proj_forward = rel @ direction
    proj_side = rel @ lateral
    mask = (np.abs(proj_forward) <= length / 2.0) & (np.abs(proj_side) <= width / 2.0)
    idx_candidates = np.where(mask)[0]
    if len(idx_candidates) == 0:
        return None, None
    idx = int(idx_candidates[0])
    return idx, poses[idx]


def pose_axes_traces(origin, forward_vec, scale=3.0):
    origin = np.asarray(origin, dtype=float)
    forward = np.asarray(forward_vec, dtype=float)
    forward[2] = 0.0
    norm = np.linalg.norm(forward[:2])
    if norm < 1e-8:
        forward = np.array([1.0, 0.0, 0.0])
    else:
        forward /= norm
    lateral = np.array([-forward[1], forward[0], 0.0])
    up = np.array([0.0, 0.0, 1.0])
    axes = [
        (forward, "Pose X", "red"),
        (lateral, "Pose Y", "green"),
        (up, "Pose Z", "blue"),
    ]
    traces = []
    for vec, name, color in axes:
        end = origin + vec * scale
        traces.append(
            go.Scatter3d(
                x=[origin[0], end[0]],
                y=[origin[1], end[1]],
                z=[origin[2], end[2]],
                mode="lines",
                name=name,
                line=dict(color=color, width=5),
                showlegend=False,
            )
        )
    return traces


In [4]:
def make_box_trace(box, show_legend=False):
    corners = np.asarray(box["corners"])
    loop = np.vstack([corners, corners[0]])
    return go.Scatter3d(
        x=loop[:, 0],
        y=loop[:, 1],
        z=loop[:, 2],
        mode="lines",
        name="Box" if show_legend else None,
        line=dict(color="orange", width=4),
        showlegend=show_legend,
    )


def box_axis_ranges(box, padding=2.0):
    corners = np.asarray(box["corners"])
    mins = corners.min(axis=0) - padding
    maxs = corners.max(axis=0) + padding
    return (mins[0], maxs[0]), (mins[1], maxs[1]), (mins[2], maxs[2])


def build_figure(box_index, zoom_box=False):
    selected_box = boxes[box_index]
    pts_in_box = filter_points_in_box(target_points, selected_box)
    pose_idx, pose_point = first_pose_index_in_box(poses, selected_box)
    next_pose_idx, next_pose_point = (None, None)
    if box_index + 1 < len(boxes):
        next_pose_idx, next_pose_point = first_pose_index_in_box(poses, boxes[box_index + 1])
    fig = go.Figure()
    fig.add_trace(
        go.Scatter3d(
            x=poses[:, 0],
            y=poses[:, 1],
            z=poses[:, 2],
            mode="lines",
            name="Trajektorie",
            line=dict(color="steelblue", width=3),
            opacity=0.4,
        )
    )
    fig.add_trace(make_box_trace(selected_box, show_legend=True))
    if len(target_points):
        fig.add_trace(
            go.Scatter3d(
                x=np.asarray(target_points)[:, 0],
                y=np.asarray(target_points)[:, 1],
                z=np.asarray(target_points)[:, 2],
                mode="markers",
                name="Alle TargetPoints",
                marker=dict(color="gray", size=3),
                opacity=0.35,
            )
        )
    if len(pts_in_box):
        fig.add_trace(
            go.Scatter3d(
                x=pts_in_box[:, 0],
                y=pts_in_box[:, 1],
                z=pts_in_box[:, 2],
                mode="markers",
                name="Treffer",
                marker=dict(color="crimson", size=6, symbol="circle"),
            )
        )
    if pose_point is not None:
        axis_length = min(selected_box["length"], 5.0)
        for trace in pose_axes_traces(pose_point, selected_box["direction"], scale=axis_length / 2.0):
            fig.add_trace(trace)
    if next_pose_point is not None:
        next_box = boxes[box_index + 1]
        axis_length = min(next_box["length"], 5.0)
        for trace in pose_axes_traces(next_pose_point, next_box["direction"], scale=axis_length / 2.0):
            fig.add_trace(trace)
    layout = dict(
        title=f"Box #{box_index} – Trefferpunkte: {len(pts_in_box)}",
        scene=dict(
            xaxis_title="X [m]",
            yaxis_title="Y [m]",
            zaxis_title="Z [m]",
            aspectmode="data",
        ),
        margin=dict(l=0, r=0, t=50, b=0),
    )
    if zoom_box:
        x_range, y_range, z_range = box_axis_ranges(selected_box)
        layout["scene"].update(
            xaxis=dict(range=list(x_range)),
            yaxis=dict(range=list(y_range)),
            zaxis=dict(range=list(z_range)),
        )
    fig.update_layout(**layout)
    return fig, pts_in_box, pose_idx, next_pose_idx


In [5]:
box_options = [(f"Box {i}", i) for i in range(len(boxes))]
box_dropdown = widgets.Dropdown(options=box_options, description="Box", value=0)
zoom_button = widgets.Button(description="Zoom to Box", button_style="info")
reset_button = widgets.Button(description="Reset Zoom")
figure_output = widgets.Output()
info_output = widgets.Output()


def render(zoom_box=False):
    with figure_output:
        figure_output.clear_output(wait=True)
        fig, pts, pose_idx, next_pose_idx = build_figure(box_dropdown.value, zoom_box=zoom_box)
        fig.show()
    with info_output:
        info_output.clear_output(wait=True)
        print(f"Box-Index: {box_dropdown.value}")
        print(f"Trefferpunkte: {len(pts)}")
        if len(pts):
            print("Erste Punkte:\n", np.round(pts[:min(5, len(pts))], 3))
        if pose_idx is not None:
            print(f"Erste Pose Index: {pose_idx}")
        else:
            print("Erste Pose Index: keine")
        if next_pose_idx is not None:
            print(f"Erste Pose nächste Box: {next_pose_idx}")
        else:
            print("Erste Pose nächste Box: keine")


def update_visualization(change):
    render(zoom_box=False)


def on_zoom_clicked(button):
    render(zoom_box=True)


def on_reset_clicked(button):
    render(zoom_box=False)


box_dropdown.observe(update_visualization, names="value")
zoom_button.on_click(on_zoom_clicked)
reset_button.on_click(on_reset_clicked)
display(widgets.HBox([box_dropdown, zoom_button, reset_button]))
display(figure_output)
display(info_output)
render(zoom_box=False)


Output()

Output()